In [1]:
import os

from openai import OpenAI

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

client = OpenAI(api_key=OPENAI_API_KEY)

completion = client.chat.completions.create(
    model='gpt-3.5-turbo-0125',
    messages=[{'role': 'user', 'content': 'hi'}],
    temperature=0.0,
)

In [2]:
import json

with open('./res/reviews.json', 'r') as f:
    review_list = json.load(f)

review_list[:3]

[{'title': '子供には最高屋上スパ',
  'review': '2家族10人で利用させて頂きました。和室湖畔側のお部屋は景色も良く清潔感あふれるお部屋でした。夕食朝食ともに大満足でした。なにより子供達が屋上スパに大大大満足でした。いつも忙しく子供との時間もとれて無かったので。とても良い旅行になりました。ありがとう感謝',
  'star': 5,
  'date': '2025年12月',
  'room': 5,
  'bath': 5,
  'breakfast': 5,
  'dinner': 5,
  'service': 5,
  'cleanliness': 5},
 {'title': '大変満足してます',
  'review': 'ご飯も美味しくて、温泉もよく、部屋もそれなりでよかった。花火も綺麗にみることができて、満足しています。また利用したいと思います。',
  'star': 4,
  'date': '2026年2月',
  'room': 4,
  'bath': 4,
  'breakfast': 4,
  'dinner': 4,
  'service': 4,
  'cleanliness': 4},
 {'title': '夜明けの天空スパはステキ',
  'review': '16時前にチェックインしたのですが既に18時台のお食事は満席でした。湖が見える良い部屋だったため20時の花火はお部屋で見たかったので、19時のお食事だとせっかくのバイキングも何となく急かされて、途中でお部屋に戻ってからまた食事席に着くのもなぁと…少し残念でした。(まだお食事中って札を置いて席を立つことは可能)夕食も朝食も種類が沢山あって、選ぶのが楽しかったです。天空スパは早朝に入りましたが、段々夜が明けていく感じがステキでした。',
  'star': 5,
  'date': '2026年2月',
  'room': 5,
  'bath': 5,
  'breakfast': 5,
  'dinner': 5,
  'service': 5,
  'cleanliness': 5}]

In [3]:
good_cnt, bad_cnt = 0, 0
for r in review_list:
    if r['star'] == 5:
        good_cnt += 1
    else:
        bad_cnt += 1

print(f'高評価（5点）: {good_cnt}件, それ以外: {bad_cnt}件')

高評価（5点）: 53件, それ以外: 97件


In [4]:
reviews_good, reviews_bad = [], []

for r in review_list:
    if r['star'] == 5:
        reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
    else:
        reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

reviews_good_text = '\n'.join(reviews_good)
reviews_bad_text = '\n'.join(reviews_bad)

In [5]:
import datetime
import re

def preprocess_reviews(path='./res/reviews.json'):
    with open(path, 'r', encoding='utf-8') as f:
        review_list = json.load(f)

    reviews_good, reviews_bad = [], []

    current_date = datetime.datetime.now()
    date_boundary = current_date - datetime.timedelta(days=6*30)

    for r in review_list:
        review_date_str = r.get('date', '')
        # 「YYYY年MM月」形式の日付をパース
        m = re.match(r'(\d{4})年(\d{1,2})月', review_date_str)
        if m:
            review_date = datetime.datetime(int(m.group(1)), int(m.group(2)), 1)
        else:
            review_date = current_date

        if review_date < date_boundary:
            continue

        if r['star'] == 5:
            reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
        else:
            reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

    reviews_good_text = '\n'.join(reviews_good)
    reviews_bad_text = '\n'.join(reviews_bad)

    return reviews_good_text, reviews_bad_text

good, bad = preprocess_reviews()

In [ ]:
def pairwise_eval(reviews, answer_a, answer_b):
    eval_prompt = f"""[System]
Please act as an impartial judge and evaluate the quality of the Japanese summaries provided by two
AI assistants to the set of user reviews on accommodations displayed below. You should choose the assistant that
follows the user’s instructions and answers the user’s question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.
[User Reviews]
{reviews}
[The Start of Assistant A’s Answer]
{answer_a}
[The End of Assistant A’s Answer]
[The Start of Assistant B’s Answer]
{answer_b}
[The End of Assistant B’s Answer]"""
    
    completion = client.chat.completions.create(
        model='gpt-4o-2024-05-13',
        messages=[{'role': 'user', 'content': eval_prompt}],
        temperature=0.0
    )

    return completion

In [33]:
PROMPT_BASELINE = f"""以下の宿泊施設のレビューを5文以内で要約してください："""

In [34]:
reviews, _ = preprocess_reviews(path='./res/reviews.json')

def summarize(reviews, prompt, temperature=0.0, model='gpt-3.5-turbo-0125'):
    prompt = prompt + '\n\n' + reviews

    completion = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature
    )

    return completion

print(summarize(reviews, PROMPT_BASELINE).choices[0].message.content)

1. 和室湖畔側のお部屋で家族10人で利用。夕食朝食も大満足。子供達は屋上スパに大満足。
2. 湖が見える部屋で18時台の食事が満席で残念。天空スパは早朝に入り、夜が明ける感じがステキ。
3. 急な骨折で和室を希望したが対応してもらえ、食事も助かった。スタッフの対応に感謝。
4. 露天風呂が素晴らしく、朝早く湖や山なみが幻想的。冬景色を楽しんだ形。
5. 料理の品数が多く、アップグレードしてもらい感謝。スタッフの対応も良かった。


In [35]:
summary_gpt4 = summarize(reviews, PROMPT_BASELINE, model='gpt-4o-2024-05-13').choices[0].message.content

In [56]:
eval_count = 10

summaries_baseline = [summarize(reviews, PROMPT_BASELINE, temperature=1.0).choices[0].message.content for _ in range(eval_count)]
summaries_baseline

['1. 和室湖畔側の部屋で2家族10人で宿泊。夕食朝食満足し、子供達は屋上スパで大満足。\n2. 18時の食事が満席で、急かされる感じが残念だったが、天空スパはステキだった。\n3. 急な骨折でベッド恐怖症にも対応し、スタッフの対応に感謝。眺めの良い部屋で楽しい時間を過ごす。\n4. 露天風呂が素晴らしく、湖や山の景色が幻想的。朝早くの入浴を楽しんだ。\n5. 食事が豊富で満足、サウナも気持ちよく何度も利用。部屋や景観は良く、次は別の季節に訪れたい。',
 '1. 和室湖畔側のお部屋で家族連れに大変満足\n2. 夕食や朝食は美味しく、子供が屋上スパで大満足\n3. 急な骨折に対応してくれ、良い景色のお部屋で静養\n4. 露天風呂が素晴らしく、朝の景色が幻想的\n5. 地元でも評判が良く、食事やバイキングが大満足',
 '1. 湖畔側の和室が景色もよく清潔感があり、子供達が屋上スパに大満足。\n2. 夕食や朝食のバイキングが豊富で満足。天空スパで早朝の湖や山を楽しんだ。\n3. スタッフの対応が素晴らしく感謝。食事や部屋、眺望が良く、静養に最適。\n4. 露天風呂が素晴らしく、湖や山を眺めるのが幻想的。朝早く行ったら幻想的な風景を楽しめた。\n5. 料理の品数が多く、全体的に満足。屋上プールからの景色が素晴らしかった。',
 '1. 家族10人で利用。和室湖畔側の清潔なお部屋。夕食朝食大満足。子供たちは屋上スパで大満足。良い旅行で感謝。\n\n2. 湖が見える良い部屋。20時の花火を部屋で見たかったが19時のお食事で急かされる感じ。食事は楽しめる。天空スパも素晴らしい。\n\n3. 急な骨折にも対応し感謝。和室を希望にも快く対応。食事やスタッフの対応に満足。子供や花火の観賞も楽しめた。\n\n4. 素晴らしい露天風呂。朝早く入ると幻想的な景色が見える。料理の品数も豊富で楽しめる。\n\n5. 冬の阿寒湖を楽しむ形。温泉やサウナも気持ちよく、料理も満足のいく品揃え。次は別の季節に再訪したい。',
 '1. 2家族10人で利用。和室湖畔側のお部屋は清潔で景色が良かった。子供達が屋上スパに大満足していた。\n2. 夕食や朝食が大満足。ただ、時間帯により混雑。天空スパで夜明けを感じられた。\n3. 急な申し出にも対応し、静養できた。眺めのよいお部屋で良い旅行となった。\n4

In [51]:
from tqdm import tqdm

def pairwise_eval_batch(reviews, answers_a, answers_b):
    a_cnt, b_cnt, draw_cnt = 0, 0, 0
    for i in tqdm(range(len(answers_a))):
        completion = pairwise_eval(reviews, answers_a[i], answers_b[i])
        verdict_text = completion.choices[0].message.content

        if '[[A]]' in verdict_text:
            a_cnt += 1
        elif '[[B]]' in verdict_text:
            b_cnt += 1
        elif '[[C]]' in verdict_text:
            draw_cnt += 1
        else:
            print('Evaluation Error')

    return a_cnt, b_cnt, draw_cnt

# gpt-3.5 + 단순 프롬프트 vs gpt-4o + 단순 프롬프트 (기존 비교)
print("=== gpt-3.5 (BASELINE) vs gpt-4o (BASELINE) ===")
wins, losses, ties = pairwise_eval_batch(reviews, summaries_baseline, [summary_gpt4 for _ in range(len(summaries_baseline))])
print(f'gpt-3.5 Wins: {wins}, gpt-4o Wins: {losses}, Ties: {ties}')

=== gpt-3.5 (BASELINE) vs gpt-4o (BASELINE) ===


100%|██████████| 10/10 [00:43<00:00,  4.33s/it]

gpt-3.5 Wins: 7, gpt-4o Wins: 3, Ties: 0


In [63]:
GEVAL_METRICS = {
    'relevance': {
        'criteria': 'Relevance - selection of important content from the source reviews.',
        'steps': """1. Read the summary and the source reviews carefully.
2. Compare the summary to the source reviews and identify the main points of the reviews.
3. Assess how well the summary covers the main points of the reviews, and how much irrelevant or redundant information it contains.
4. Assign a relevance score from 1 to 5."""
    },
    'coherence': {
        'criteria': 'Coherence - the collective quality of all sentences in the summary.',
        'steps': """1. Read the summary carefully and identify if it presents a well-organized structure.
2. Check if the summary flows naturally from one sentence to the next, with logical connections.
3. Assess whether the summary reads as a unified piece rather than a collection of disconnected sentences.
4. Assign a coherence score from 1 to 5."""
    },
    'consistency': {
        'criteria': 'Consistency - the factual alignment between the summary and the source reviews.',
        'steps': """1. Read the summary and the source reviews carefully.
2. Identify any facts stated in the summary that are not present in or contradict the source reviews.
3. A summary that contains no hallucinated or contradictory facts should receive a high score.
4. Assign a consistency score from 1 to 5."""
    },
    'fluency': {
        'criteria': 'Fluency - the quality of the summary in terms of grammar, spelling, punctuation, word choice, and sentence structure.',
        'steps': """1. Read the summary and check for any grammatical errors, awkward phrasing, or unnatural expressions.
2. Assess whether the summary uses appropriate language and tone.
3. Check if the summary is easy to read and understand.
4. Assign a fluency score from 1 to 5."""
    }
}

def geval_score(reviews, summary, metric_name):
    metric = GEVAL_METRICS[metric_name]
    prompt = f"""You will be given a set of user reviews of an accommodation and a summary written by an AI assistant.
Your task is to rate the summary on one specific metric.
Please make sure you read and understand these instructions carefully. Please keep this document open while reviewing, and refer to it as needed.

Evaluation Criteria:
{metric['criteria']}

Evaluation Steps:
{metric['steps']}

Source Reviews:
{reviews}

Summary:
{summary}

Evaluation Form (scores ONLY):
- {metric_name} (1-5):"""

    completion = client.chat.completions.create(
        model='gpt-4o-2024-05-13',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=5
    )

    try:
        score = int(completion.choices[0].message.content.strip().split('\n')[0].strip())
        return min(max(score, 1), 5)
    except (ValueError, IndexError):
        text = completion.choices[0].message.content.strip()
        for c in text:
            if c.isdigit() and 1 <= int(c) <= 5:
                return int(c)
        return None

def geval_all_metrics(reviews, summary):
    scores = {}
    for metric_name in GEVAL_METRICS:
        scores[metric_name] = geval_score(reviews, summary, metric_name)
    return scores

def geval_batch(reviews, summaries):
    all_scores = {m: [] for m in GEVAL_METRICS}
    for i, summary in enumerate(tqdm(summaries)):
        scores = geval_all_metrics(reviews, summary)
        for m, s in scores.items():
            all_scores[m].append(s)
    
    print("\n=== G-Eval Results ===")
    for m in GEVAL_METRICS:
        valid = [s for s in all_scores[m] if s is not None]
        avg = sum(valid) / len(valid) if valid else 0
        print(f"{m:>15}: {avg:.2f} (scores: {valid})")
    return all_scores

In [64]:
# 디버깅: 실제 API 응답 확인
metric = GEVAL_METRICS['relevance']
prompt = f"""You will be given a set of user reviews of an accommodation and a summary written by an AI assistant.
Your task is to rate the summary on one specific metric.
Please make sure you read and understand these instructions carefully. Please keep this document open while reviewing, and refer to it as needed.

Evaluation Criteria:
{metric['criteria']}

Evaluation Steps:
{metric['steps']}

Source Reviews:
{reviews}

Summary:
{summary_gpt4}

Evaluation Form (scores ONLY):
- relevance (1-5):"""

completion = client.chat.completions.create(
    model='gpt-4o-2024-05-13',
    messages=[{'role': 'user', 'content': prompt}],
    temperature=0.0,
    max_tokens=5
)

print(repr(completion.choices[0].message.content))

'- relevance (1-'


In [52]:
prompt = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。
    
以下の宿泊施設のレビューを要約してください："""

eval_count = 10
summaries = [summarize(reviews, prompt, temperature=1.0).choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:42<00:00,  4.21s/it]

Wins: 6, Losses: 4, Ties: 0


In [53]:
import datetime
from dateutil import parser

def preprocess_reviews(path='./res/reviews.json'):
    with open(path, 'r', encoding='utf-8') as f:
        review_list = json.load(f)

    reviews_good, reviews_bad = [], []

    current_date = datetime.datetime.now()
    date_boundary = current_date - datetime.timedelta(days=6*30)

    filtered_cnt = 0
    for r in review_list:
        review_date_str = r['date']
        try:
            review_date = parser.parse(review_date_str)
        except (ValueError, TypeError):
            review_date = current_date

        if review_date < date_boundary:
            continue
        if len(r['review']) < 30:
            filtered_cnt += 1
            continue

        if r['star'] == 5:
            reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
        else:
            reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

    reviews_good = reviews_good[:min(len(reviews_good), 30)]
    reviews_bad = reviews_bad[:min(len(reviews_bad), 30)]

    reviews_good_text = '\n'.join(reviews_good)
    reviews_bad_text = '\n'.join(reviews_bad)

    return reviews_good_text, reviews_bad_text

reviews, _ = preprocess_reviews()

In [54]:
reviews_1shot, _ = preprocess_reviews(path='./res/reviews.json')
summary_1shot = summarize(reviews_1shot, prompt, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content
prompt_1shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。

以下はレビューと要約の例です。
例のレビュー：
{reviews_1shot}
例の要約結果：
{summary_1shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:46<00:00,  4.64s/it]

Wins: 6, Losses: 4, Ties: 0


In [55]:
prompt_1shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。以下はレビューと要約の例です。
例のレビュー：
{reviews_1shot}
例の要約結果：
{summary_1shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt_1shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:51<00:00,  5.15s/it]

Wins: 3, Losses: 7, Ties: 0


In [18]:
reviews_1shot, _ = preprocess_reviews('./res/kunosato.json')
summary_1shot = summarize(reviews_1shot, prompt, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content

In [19]:
reviews_2shot, _ = preprocess_reviews('./res/global_view.json')
summary_2shot = summarize(reviews_2shot, prompt_1shot, temperature=0.0, model='gpt-4-turbo-2024-04-09').choices[0].message.content

prompt_2shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。以下はレビューと要約の例です。

例のレビュー 1：
{reviews_1shot}
例の要約結果 1：
{summary_1shot}

例のレビュー 2：
{reviews_2shot}
例の要約結果 2：
{summary_2shot}
    
以下の宿泊施設のレビューを要約してください："""

summaries = [summarize(reviews, prompt_2shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content for _ in range(eval_count)]
wins, losses, ties = pairwise_eval_batch(reviews, summaries, [summary_gpt4 for _ in range(len(summaries))])
print(f'Wins: {wins}, Losses: {losses}, Ties: {ties}')

100%|██████████| 10/10 [00:54<00:00,  5.46s/it]

Wins: 3, Losses: 7, Ties: 0
